# 🔍 Exploration des agents hors barème

Ce notebook analyse les agents dont le salaire se situe en dehors des bornes min/max du référentiel ANSTAT 2025.

**Source** : `s3://gold/panel_admin/base_avec_statut.parquet` (produit par l'étape 07)

**Rappel** : Chaque ligne = un agent × un mois. Un même agent peut apparaître plusieurs fois.

## 1. Chargement des données

In [4]:
import io
import os
import warnings
warnings.filterwarnings('ignore')

from dotenv import load_dotenv
import boto3
from botocore.client import Config
import polars as pl

load_dotenv(".env")

s3 = boto3.client(
    "s3",
    endpoint_url          = os.getenv("MINIO_ENDPOINT", "http://192.168.1.230:30137"),
    aws_access_key_id     = os.getenv("MINIO_ACCESS_KEY", "datalab-team"),
    aws_secret_access_key = os.getenv("MINIO_SECRET_KEY", "minio-datalabteam123"),
    config                = Config(signature_version="s3v4"),
    region_name           = "us-east-1",
    verify                = False,
)

# Charger base_avec_statut.parquet (TOUS les agents, conformes et hors barème)
buf = io.BytesIO()
s3.download_fileobj("gold", "panel_admin/base_avec_statut.parquet", buf)
buf.seek(0)
base = pl.read_parquet(buf)

print(f"Base complète : {len(base):,} lignes × {len(base.columns)} colonnes")
print(f"Colonnes : {base.columns}")

base.head(10)

Base complète : 7,589,254 lignes × 55 colonnes
Colonnes : ['cle_unique', 'matricule', 'nom', 'sexe', 'date_naissance', 'situation_matrimoniale', 'nombre_enfant', 'situation_normalisee', 'CODE_ORGANISME', 'organisme', 'CODE_EMPLOI', 'emploi', 'Code_CITP', 'Metier_CITP', 'CODE_GRADE', 'grade', 'statut_fonctionnaire', 'CODE_FONCTION', 'fonction', 'CODE_SERVICE', 'service', 'CODE_AFFECTATION', 'lieu_affectation', 'CODE_POSTE', 'prise_service', 'date_retraite', 'age_retraite', 'GRADE', 'GRADE_SOURCE', 'mois_annee', 'annee', 'mois', 'montant_brut', 'montant_net', 'retenue_pension', 'impot', 'charge_patronale', 'montant_brut_SOURCE', 'montant_net_SOURCE', 'fichier_source', 'NB_POSTES', 'montant_brut_total', 'montant_brut_bareme', 'montant_net_total', 'ANNEE', 'MOIS_NUM', 'MOIS', 'SALAIRE_BRUT', 'SEXE_STD', 'bareme', 'grade_ref', 'bareme_detail', 'min_bareme', 'max_bareme', 'Statut']


cle_unique,matricule,nom,sexe,date_naissance,situation_matrimoniale,nombre_enfant,situation_normalisee,CODE_ORGANISME,organisme,CODE_EMPLOI,emploi,Code_CITP,Metier_CITP,CODE_GRADE,grade,statut_fonctionnaire,CODE_FONCTION,fonction,CODE_SERVICE,service,CODE_AFFECTATION,lieu_affectation,CODE_POSTE,prise_service,date_retraite,age_retraite,GRADE,GRADE_SOURCE,mois_annee,annee,mois,montant_brut,montant_net,retenue_pension,impot,charge_patronale,montant_brut_SOURCE,montant_net_SOURCE,fichier_source,NB_POSTES,montant_brut_total,montant_brut_bareme,montant_net_total,ANNEE,MOIS_NUM,MOIS,SALAIRE_BRUT,SEXE_STD,bareme,grade_ref,bareme_detail,min_bareme,max_bareme,Statut
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i32,i32,f64,f64,f64,f64,f64,str,str,str,u32,f64,f64,f64,i32,i32,str,f64,str,str,str,str,f64,f64,str
"""057426N15""","""057426N""","""CISSOKO MOUSSA""","""Masculin""","""1946-03-01""","""Célibataire""","""0""","""regul_indemnites""","""15""","""Min. d'Etat Administration du …","""9DAF""","""Administrateur Civil""","""1112""","""Hauts fonctionnaires""","""33""","""1ère Classe 3ème Echelon""","""Fonctionnaire Catégorie A7""",null,""" ""","""1514""","""D. G. A. A. T.""","""4DA""","""Gagnoa""",null,"""1987-01-03""",null,null,"""A7""","""OBSERVE""","""092024""",2024,9,810000.0,810000.0,0.0,0.0,0.0,"""OBSERVE""","""OBSERVE""","""092024.xlsx""",1,810000.0,0.0,810000.0,2024,9,"""Septembre""",0.0,"""Homme""","""Barême Général""","""A4""","""Barême Général""",646210.0,750915.0,"""Inférieur au minimum du barème"""
"""084797N34""","""084797N""","""KOUAKOU YAO""","""Masculin""","""1952-01-01""","""Marié""","""3""","""regul_indemnites""","""34""","""Min. Enseignement Technique, F…","""25ET""","""Professeur de Lycée Ensgt Tech…","""2330""","""Enseignants du secondaire""","""21""","""Classe Principale 1er Echelon""","""Fonctionnaire Catégorie A4""",null,""" ""","""1911""","""Direction des Affaires Adminis…","""3AA""","""Bouaké""",null,"""1980-10-01""",null,null,"""A4""","""OBSERVE""","""012024""",2024,1,3.221702e6,3.201702e6,0.0,0.0,0.0,"""OBSERVE""","""OBSERVE""","""012024.xlsx""",1,3.221702e6,0.0,3.201702e6,2024,1,"""Janvier""",0.0,"""Homme""",null,null,null,null,null,"""Cluster non déterminé"""
"""093833B38""","""093833B""","""OUATTARA FOUSSENI""","""Masculin""","""1951-11-01""","""Célibataire""","""2""","""regul_indemnites""","""38""","""Min. de l'Intérieur et de la S…","""A2DD""","""Commissaire de Police""","""1112""","""Hauts fonctionnaires""","""LQ""","""Commissaire principal 1er éch""","""Fonctionnaire Catégorie A7""",null,""" ""","""3813""","""Dir Personnel Police Nationale""","""1AA""","""Abidjan""",null,"""1975-10-01""",null,null,"""A7""","""OBSERVE""","""022024""",2024,2,1.62e6,1.62e6,0.0,0.0,0.0,"""OBSERVE""","""OBSERVE""","""022024.xlsx""",1,1.62e6,0.0,1.62e6,2024,2,"""Février""",0.0,"""Homme""","""para militaire""","""--""","""police""",666210.0,770915.0,"""Inférieur au minimum du barème"""
"""093833B38""","""093833B""","""OUATTARA FOUSSENI""","""Masculin""","""1951-11-01""","""Célibataire""","""2""","""regul_indemnites""","""38""","""Min. de l'Intérieur et de la S…","""A2DD""","""Commissaire de Police""","""1112""","""Hauts fonctionnaires""","""LQ""","""Commissaire principal 1er éch""","""Fonctionnaire Catégorie A7""",null,""" ""","""3813""","""Dir Personnel Police Nationale""","""1AA""","""Abidjan""",null,"""1975-10-01""",null,null,"""A7""","""OBSERVE""","""052024""",2024,5,1.62e6,1.62e6,0.0,0.0,0.0,"""OBSERVE""","""OBSERVE""","""052024.xlsx""",1,1.62e6,0.0,1.62e6,2024,5,"""Mai""",0.0,"""Homme""","""para militaire""","""--""","""police""",666210.0,770915.0,"""Inférieur au minimum du barème"""
"""093833B38""","""093833B""","""OUATTARA FOUSSENI""","""Masculin""","""1951-11-01""","""Célibataire""","""2""","""regul_indemnites""","""38""","""Min. de l'Intérieur et de la S…","""A2DD""","""Commissaire de Police""","""1112""","""Hauts fonctionnaires""","""LQ""","""

## 2. Vue d'ensemble : lignes vs agents uniques

In [5]:
# Distribution des statuts (en lignes)
print("=" * 60)
print("DISTRIBUTION DES STATUTS (en lignes agent × mois)")
print("=" * 60)

n_total = len(base)
statut_counts = base["Statut"].value_counts().sort("count", descending=True)
for row in statut_counts.iter_rows(named=True):
    pct = 100 * row["count"] / n_total
    print(f"  {row['Statut']:<40} : {row['count']:>10,}  ({pct:.1f}%)")
print(f"  {'TOTAL':<40} : {n_total:>10,}")

DISTRIBUTION DES STATUTS (en lignes agent × mois)
  Conforme au barème                       :  5,724,312  (75.4%)
  Inférieur au minimum du barème           :  1,492,729  (19.7%)
  Supérieur au maximum du barème           :    363,268  (4.8%)
  Cluster non déterminé                    :      8,945  (0.1%)
  TOTAL                                    :  7,589,254


In [6]:
# Identifier la colonne d'identifiant unique
col_id = "matricule" if "matricule" in base.columns else "cle_unique"
print(f"Identifiant utilisé : {col_id}")

# Agents UNIQUES par statut
print("\n" + "=" * 60)
print("DISTRIBUTION DES STATUTS (en agents UNIQUES)")
print("=" * 60)

agents_par_statut = (
    base.group_by("Statut")
    .agg([
        pl.col(col_id).n_unique().alias("Agents_uniques"),
        pl.len().alias("Nb_lignes"),
    ])
    .with_columns(
        (pl.col("Nb_lignes") / pl.col("Agents_uniques")).round(1).alias("Mois_moyen_par_agent")
    )
    .sort("Agents_uniques", descending=True)
)
print(agents_par_statut)

Identifiant utilisé : matricule

DISTRIBUTION DES STATUTS (en agents UNIQUES)
shape: (4, 4)
┌────────────────────────────────┬────────────────┬───────────┬──────────────────────┐
│ Statut                         ┆ Agents_uniques ┆ Nb_lignes ┆ Mois_moyen_par_agent │
│ ---                            ┆ ---            ┆ ---       ┆ ---                  │
│ str                            ┆ u32            ┆ u32       ┆ f64                  │
╞════════════════════════════════╪════════════════╪═══════════╪══════════════════════╡
│ Conforme au barème             ┆ 289744         ┆ 5724312   ┆ 19.8                 │
│ Supérieur au maximum du barème ┆ 135159         ┆ 363268    ┆ 2.7                  │
│ Inférieur au minimum du barème ┆ 114706         ┆ 1492729   ┆ 13.0                 │
│ Cluster non déterminé          ┆ 592            ┆ 8945      ┆ 15.1                 │
└────────────────────────────────┴────────────────┴───────────┴──────────────────────┘


In [13]:
# Un même agent peut-il être PARFOIS conforme et PARFOIS hors barème ?
print("=" * 50)
print("AGENTS PARFOIS CONFORME ET PARFOIS HORS BAREME")
print("=" * 50)

statuts_par_agent = (
    base.group_by(col_id)
    .agg(pl.col("Statut").n_unique().alias("nb_statuts_differents"))
)

n_multi_statut = int((statuts_par_agent["nb_statuts_differents"] > 1).sum())
n_agents_total = len(statuts_par_agent)

print(f"Agents avec un seul statut    : {n_agents_total - n_multi_statut:,}")
print(f"Agents avec plusieurs statuts : {n_multi_statut:,}  "
      f"({100*n_multi_statut/n_agents_total:.1f}%)")
print("\n→ Ces agents changent de catégorie selon les mois (salaire variable)")

AGENTS PARFOIS CONFORME ET PARFOIS HORS BAREME
Agents avec un seul statut    : 186,716
Agents avec plusieurs statuts : 160,316  (46.2%)

→ Ces agents changent de catégorie selon les mois (salaire variable)


## 3. Filtrage : agents hors barème uniquement

In [5]:
# Filtrer les hors barème
hors = base.filter(
    pl.col("Statut").is_in([
        "Inférieur au minimum du barème",
        "Supérieur au maximum du barème",
    ])
)

print(f"Lignes hors barème    : {len(hors):,}")
print(f"Agents uniques        : {hors[col_id].n_unique():,}")

# Détecter la colonne salaire
col_sal = next((c for c in ["montant_brut_bareme", "SALAIRE_BRUT", "montant_brut"] 
                if c in hors.columns), None)
print(f"Colonne salaire       : {col_sal}")

# Aperçu
print("\nAperçu des données :")
cols_apercu = [c for c in [col_id, "mois_annee", "ANNEE", col_sal, 
               "min_bareme", "max_bareme", "Statut", "GRADE", "bareme"]
               if c and c in hors.columns]
hors.select(cols_apercu).head(10)

Lignes hors barème    : 1,855,997
Agents uniques        : 188,772
Colonne salaire       : montant_brut_bareme

Aperçu des données :


matricule,mois_annee,ANNEE,montant_brut_bareme,min_bareme,max_bareme,Statut,GRADE,bareme
str,str,i32,f64,f64,f64,str,str,str
"""057426N""","""092024""",2024,0.0,646210.0,750915.0,"""Inférieur au minimum du barème""","""A7""","""Barême Général"""
"""093833B""","""022024""",2024,0.0,666210.0,770915.0,"""Inférieur au minimum du barème""","""A7""","""para militaire"""
"""093833B""","""052024""",2024,0.0,666210.0,770915.0,"""Inférieur au minimum du barème""","""A7""","""para militaire"""
"""093833B""","""082024""",2024,0.0,666210.0,770915.0,"""Inférieur au minimum du barème""","""A7""","""para militaire"""
"""093833B""","""112024""",2024,0.0,666210.0,770915.0,"""Inférieur au minimum du barème""","""A7""","""para militaire"""
"""095112A""","""022024""",2024,0.0,666210.0,770915.0,"""Inférieur au minimum du barème""","""A7""","""para militaire"""
"""095112A""","""052024""",2024,0.0,666210.0,770915.0,"""Inférieur au minimum du barème""","""A7""","""para militaire"""
"""095112A""","""082024""",2024,0.0,666210.0,770915.0,"""Inférieur au minimum du barème""","""A7""","""para militaire"""
"""095112A""","""112024""",2024,0.0,666210.0,770915.0,"""Inférieur au minimum du barème""","""A7""","""para militaire"""


## 4. Analyse des écarts au barème

**De combien les agents dépassent-ils les bornes ?**

In [6]:
# Calculer l'écart au barème
if col_sal and "min_bareme" in hors.columns and "max_bareme" in hors.columns:
    hors = hors.with_columns([
        # Écart en FCFA
        pl.when(pl.col("Statut") == "Inférieur au minimum du barème")
          .then(pl.col(col_sal) - pl.col("min_bareme"))
          .otherwise(pl.col(col_sal) - pl.col("max_bareme"))
          .alias("ecart_fcfa"),
        # Écart en %
        pl.when(pl.col("Statut") == "Inférieur au minimum du barème")
          .then(((pl.col(col_sal) - pl.col("min_bareme")) / pl.col("min_bareme") * 100).round(1))
          .otherwise(((pl.col(col_sal) - pl.col("max_bareme")) / pl.col("max_bareme") * 100).round(1))
          .alias("ecart_pct"),
    ])
    
    print("Écart au barème par statut :")
    print(
        hors.group_by("Statut")
        .agg([
            pl.len().alias("Nb_lignes"),
            pl.col("ecart_fcfa").mean().round(0).alias("Ecart_moyen_FCFA"),
            pl.col("ecart_fcfa").median().round(0).alias("Ecart_median_FCFA"),
            pl.col("ecart_pct").mean().round(1).alias("Ecart_moyen_%"),
            pl.col("ecart_pct").median().round(1).alias("Ecart_median_%"),
        ])
        .sort("Statut")
    )

Écart au barème par statut :
shape: (2, 6)
┌─────────────────┬───────────┬─────────────────┬─────────────────┬───────────────┬────────────────┐
│ Statut          ┆ Nb_lignes ┆ Ecart_moyen_FCF ┆ Ecart_median_FC ┆ Ecart_moyen_% ┆ Ecart_median_% │
│ ---             ┆ ---       ┆ A               ┆ FA              ┆ ---           ┆ ---            │
│ str             ┆ u32       ┆ ---             ┆ ---             ┆ f64           ┆ f64            │
│                 ┆           ┆ f64             ┆ f64             ┆               ┆                │
╞═════════════════╪═══════════╪═════════════════╪═════════════════╪═══════════════╪════════════════╡
│ Inférieur au    ┆ 1492729   ┆ -39405.0        ┆ -14998.0        ┆ -10.1         ┆ -4.7           │
│ minimum du      ┆           ┆                 ┆                 ┆               ┆                │
│ barème          ┆           ┆                 ┆                 ┆               ┆                │
│ Supérieur au    ┆ 363268    ┆ 426298.0        

In [7]:
# Distribution des écarts en pourcentage
print("Distribution des écarts (%) — Inférieur au minimum :")
inferieurs = hors.filter(pl.col("Statut") == "Inférieur au minimum du barème")
if len(inferieurs) > 0:
    print(
        inferieurs.select(
            pl.col("ecart_pct").quantile(0.05).round(1).alias("P5"),
            pl.col("ecart_pct").quantile(0.10).round(1).alias("P10"),
            pl.col("ecart_pct").quantile(0.25).round(1).alias("P25"),
            pl.col("ecart_pct").median().round(1).alias("Médiane"),
            pl.col("ecart_pct").quantile(0.75).round(1).alias("P75"),
            pl.col("ecart_pct").quantile(0.90).round(1).alias("P90"),
            pl.col("ecart_pct").quantile(0.95).round(1).alias("P95"),
        )
    )

print("\nDistribution des écarts (%) — Supérieur au maximum :")
superieurs = hors.filter(pl.col("Statut") == "Supérieur au maximum du barème")
if len(superieurs) > 0:
    print(
        superieurs.select(
            pl.col("ecart_pct").quantile(0.05).round(1).alias("P5"),
            pl.col("ecart_pct").quantile(0.10).round(1).alias("P10"),
            pl.col("ecart_pct").quantile(0.25).round(1).alias("P25"),
            pl.col("ecart_pct").median().round(1).alias("Médiane"),
            pl.col("ecart_pct").quantile(0.75).round(1).alias("P75"),
            pl.col("ecart_pct").quantile(0.90).round(1).alias("P90"),
            pl.col("ecart_pct").quantile(0.95).round(1).alias("P95"),
        )
    )

Distribution des écarts (%) — Inférieur au minimum :
shape: (1, 7)
┌───────┬───────┬──────┬─────────┬──────┬──────┬──────┐
│ P5    ┆ P10   ┆ P25  ┆ Médiane ┆ P75  ┆ P90  ┆ P95  │
│ ---   ┆ ---   ┆ ---  ┆ ---     ┆ ---  ┆ ---  ┆ ---  │
│ f64   ┆ f64   ┆ f64  ┆ f64     ┆ f64  ┆ f64  ┆ f64  │
╞═══════╪═══════╪══════╪═════════╪══════╪══════╪══════╡
│ -43.2 ┆ -38.8 ┆ -9.7 ┆ -4.7    ┆ -2.4 ┆ -0.9 ┆ -0.6 │
└───────┴───────┴──────┴─────────┴──────┴──────┴──────┘

Distribution des écarts (%) — Supérieur au maximum :
shape: (1, 7)
┌─────┬─────┬─────┬─────────┬──────┬───────┬───────┐
│ P5  ┆ P10 ┆ P25 ┆ Médiane ┆ P75  ┆ P90   ┆ P95   │
│ --- ┆ --- ┆ --- ┆ ---     ┆ ---  ┆ ---   ┆ ---   │
│ f64 ┆ f64 ┆ f64 ┆ f64     ┆ f64  ┆ f64   ┆ f64   │
╞═════╪═════╪═════╪═════════╪══════╪═══════╪═══════╡
│ 0.0 ┆ 0.4 ┆ 1.1 ┆ 3.4     ┆ 31.8 ┆ 326.9 ┆ 560.9 │
└─────┴─────┴─────┴─────────┴──────┴───────┴───────┘


In [9]:
# Tranches d'écart : regrouper les agents par amplitude d'écart
if "ecart_pct" in hors.columns:
    hors_avec_tranche = hors.with_columns(
        pl.when(pl.col("ecart_pct").abs() <= 5)
          .then(pl.lit("1. Écart ≤ 5%"))
        .when(pl.col("ecart_pct").abs() <= 15)
          .then(pl.lit("2. Écart 5-15%"))
        .when(pl.col("ecart_pct").abs() <= 30)
          .then(pl.lit("3. Écart 15-30%"))
        .when(pl.col("ecart_pct").abs() <= 50)
          .then(pl.lit("4. Écart 30-50%"))
        .otherwise(pl.lit("5. Écart > 50%"))
        .alias("tranche_ecart")
    )
    
    print("Répartition par tranche d'écart :")
    tranches = (
        hors_avec_tranche
        .group_by(["Statut", "tranche_ecart"])
        .agg([
            pl.len().alias("Nb_lignes"),
            pl.col(col_id).n_unique().alias("Agents_uniques"),
        ])
        .sort(["Statut", "tranche_ecart"])
    )
    print(tranches)
    
    print("\n→ Les écarts ≤ 5% peuvent être des arrondis ou des cas limites.")
    print("→ Les écarts > 50% sont probablement des erreurs de saisie ou des cas spéciaux.")

Répartition par tranche d'écart :
shape: (10, 4)
┌────────────────────────────────┬─────────────────┬───────────┬────────────────┐
│ Statut                         ┆ tranche_ecart   ┆ Nb_lignes ┆ Agents_uniques │
│ ---                            ┆ ---             ┆ ---       ┆ ---            │
│ str                            ┆ str             ┆ u32       ┆ u32            │
╞════════════════════════════════╪═════════════════╪═══════════╪════════════════╡
│ Inférieur au minimum du barème ┆ 1. Écart ≤ 5%   ┆ 794939    ┆ 76330          │
│ Inférieur au minimum du barème ┆ 2. Écart 5-15%  ┆ 449934    ┆ 45558          │
│ Inférieur au minimum du barème ┆ 3. Écart 15-30% ┆ 80543     ┆ 7177           │
│ Inférieur au minimum du barème ┆ 4. Écart 30-50% ┆ 148924    ┆ 13790          │
│ Inférieur au minimum du barème ┆ 5. Écart > 50%  ┆ 18389     ┆ 5582           │
│ Supérieur au maximum du barème ┆ 1. Écart ≤ 5%   ┆ 210916    ┆ 30797          │
│ Supérieur au maximum du barème ┆ 2. Écart 5-15%

## 5. Analyse par grade

**Quels grades sont les plus touchés ?**

In [10]:
if "GRADE" in hors.columns:
    print("Hors barème par Grade :")
    stats_grade = (
        hors.group_by(["Statut", "GRADE"])
        .agg([
            pl.len().alias("Nb_lignes"),
            pl.col(col_id).n_unique().alias("Agents_uniques"),
            pl.col(col_sal).mean().round(0).alias("Salaire_moyen"),
            pl.col(col_sal).median().round(0).alias("Salaire_median"),
            pl.col("min_bareme").first().round(0).alias("Borne_min"),
            pl.col("max_bareme").first().round(0).alias("Borne_max"),
        ])
        .sort(["Statut", "Agents_uniques"], descending=[False, True])
    )
    print(stats_grade)

Hors barème par Grade :
shape: (23, 8)
┌─────────────┬───────┬───────────┬─────────────┬────────────┬────────────┬───────────┬────────────┐
│ Statut      ┆ GRADE ┆ Nb_lignes ┆ Agents_uniq ┆ Salaire_mo ┆ Salaire_me ┆ Borne_min ┆ Borne_max  │
│ ---         ┆ ---   ┆ ---       ┆ ues         ┆ yen        ┆ dian       ┆ ---       ┆ ---        │
│ str         ┆ str   ┆ u32       ┆ ---         ┆ ---        ┆ ---        ┆ f64       ┆ f64        │
│             ┆       ┆           ┆ u32         ┆ f64        ┆ f64        ┆           ┆            │
╞═════════════╪═══════╪═══════════╪═════════════╪════════════╪════════════╪═══════════╪════════════╡
│ Inférieur   ┆ C3    ┆ 627782    ┆ 42922       ┆ 228988.0   ┆ 259276.0   ┆ 230210.0  ┆ 316122.0   │
│ au minimum  ┆       ┆           ┆             ┆            ┆            ┆           ┆            │
│ du barème   ┆       ┆           ┆             ┆            ┆            ┆           ┆            │
│ Inférieur   ┆ B3    ┆ 272982    ┆ 26590       ┆ 27

## 6. Analyse par secteur (barème)

**Quels secteurs sont les plus touchés ?**

In [11]:
if "bareme" in hors.columns:
    print("Hors barème par Barème/Secteur :")
    stats_bareme = (
        hors.filter(pl.col("bareme").is_not_null())
        .group_by(["Statut", "bareme"])
        .agg([
            pl.len().alias("Nb_lignes"),
            pl.col(col_id).n_unique().alias("Agents_uniques"),
            pl.col(col_sal).mean().round(0).alias("Salaire_moyen"),
        ])
        .sort(["Statut", "Agents_uniques"], descending=[False, True])
    )
    print(stats_bareme)
else:
    print("Colonne 'bareme' non disponible")

Hors barème par Barème/Secteur :
shape: (24, 5)
┌──────────────────────────┬──────────────────────┬───────────┬────────────────┬───────────────┐
│ Statut                   ┆ bareme               ┆ Nb_lignes ┆ Agents_uniques ┆ Salaire_moyen │
│ ---                      ┆ ---                  ┆ ---       ┆ ---            ┆ ---           │
│ str                      ┆ str                  ┆ u32       ┆ u32            ┆ f64           │
╞══════════════════════════╪══════════════════════╪═══════════╪════════════════╪═══════════════╡
│ Inférieur au minimum du  ┆ ENS SEC & PRIM       ┆ 506629    ┆ 43819          ┆ 316909.0      │
│ barème                   ┆                      ┆           ┆                ┆               │
│ Inférieur au minimum du  ┆ para militaire       ┆ 414345    ┆ 23595          ┆ 243611.0      │
│ barème                   ┆                      ┆           ┆                ┆               │
│ Inférieur au minimum du  ┆ Barême Général       ┆ 242811    ┆ 19615          

## 7. Analyse croisée secteur × grade

**Quels clusters (secteur × grade) posent le plus de problèmes ?**

In [12]:
if "bareme" in hors.columns and "GRADE" in hors.columns:
    print("Top 20 des clusters avec le plus d'agents hors barème :")
    clusters = (
        hors.filter(pl.col("bareme").is_not_null())
        .group_by(["bareme", "GRADE", "Statut"])
        .agg([
            pl.col(col_id).n_unique().alias("Agents_uniques"),
            pl.col(col_sal).mean().round(0).alias("Salaire_moyen"),
            pl.col(col_sal).median().round(0).alias("Salaire_median"),
            pl.col("min_bareme").first().round(0).alias("Borne_min_ref"),
            pl.col("max_bareme").first().round(0).alias("Borne_max_ref"),
        ])
        .sort("Agents_uniques", descending=True)
        .head(20)
    )
    print(clusters)

Top 20 des clusters avec le plus d'agents hors barème :
shape: (20, 8)
┌────────────┬───────┬────────────┬────────────┬────────────┬────────────┬────────────┬────────────┐
│ bareme     ┆ GRADE ┆ Statut     ┆ Agents_uni ┆ Salaire_mo ┆ Salaire_me ┆ Borne_min_ ┆ Borne_max_ │
│ ---        ┆ ---   ┆ ---        ┆ ques       ┆ yen        ┆ dian       ┆ ref        ┆ ref        │
│ str        ┆ str   ┆ str        ┆ ---        ┆ ---        ┆ ---        ┆ ---        ┆ ---        │
│            ┆       ┆            ┆ u32        ┆ f64        ┆ f64        ┆ f64        ┆ f64        │
╞════════════╪═══════╪════════════╪════════════╪════════════╪════════════╪════════════╪════════════╡
│ ENS SEC &  ┆ B3    ┆ Supérieur  ┆ 27511      ┆ 673823.0   ┆ 567146.0   ┆ 357132.0   ┆ 557145.0   │
│ PRIM       ┆       ┆ au maximum ┆            ┆            ┆            ┆            ┆            │
│            ┆       ┆ du barème  ┆            ┆            ┆            ┆            ┆            │
│ ENS SEC &  ┆ C3   

## 8. Analyse temporelle

**L'incohérence est-elle constante ou variable dans le temps ?**

In [13]:
if "ANNEE" in hors.columns:
    col_mois = "MOIS_NUM" if "MOIS_NUM" in hors.columns else "MOIS"
    
    print("Évolution par année :")
    evol_annee = (
        hors.group_by(["ANNEE", "Statut"])
        .agg([
            pl.len().alias("Nb_lignes"),
            pl.col(col_id).n_unique().alias("Agents_uniques"),
            pl.col(col_sal).mean().round(0).alias("Salaire_moyen"),
        ])
        .sort(["ANNEE", "Statut"])
    )
    print(evol_annee)
    
    if col_mois in hors.columns:
        print("\nÉvolution par mois (dernière année) :")
        derniere_annee = hors["ANNEE"].max()
        evol_mois = (
            hors.filter(pl.col("ANNEE") == derniere_annee)
            .group_by([col_mois, "Statut"])
            .agg([
                pl.col(col_id).n_unique().alias("Agents_uniques"),
            ])
            .sort([col_mois, "Statut"])
        )
        print(evol_mois)

Évolution par année :
shape: (4, 5)
┌───────┬────────────────────────────────┬───────────┬────────────────┬───────────────┐
│ ANNEE ┆ Statut                         ┆ Nb_lignes ┆ Agents_uniques ┆ Salaire_moyen │
│ ---   ┆ ---                            ┆ ---       ┆ ---            ┆ ---           │
│ i32   ┆ str                            ┆ u32       ┆ u32            ┆ f64           │
╞═══════╪════════════════════════════════╪═══════════╪════════════════╪═══════════════╡
│ 2024  ┆ Inférieur au minimum du barème ┆ 726746    ┆ 86267          ┆ 283103.0      │
│ 2024  ┆ Supérieur au maximum du barème ┆ 169065    ┆ 70029          ┆ 1.018764e6    │
│ 2025  ┆ Inférieur au minimum du barème ┆ 765983    ┆ 88780          ┆ 275261.0      │
│ 2025  ┆ Supérieur au maximum du barème ┆ 194203    ┆ 82241          ┆ 916314.0      │
└───────┴────────────────────────────────┴───────────┴────────────────┴───────────────┘

Évolution par mois (dernière année) :
shape: (24, 3)
┌──────────┬──────────────────

## 9. Analyse par sexe

In [14]:
col_sexe = next((c for c in ["SEXE_STD", "sexe"] if c in hors.columns), None)

if col_sexe:
    print("Hors barème par sexe :")
    stats_sexe = (
        hors.group_by(["Statut", col_sexe])
        .agg([
            pl.len().alias("Nb_lignes"),
            pl.col(col_id).n_unique().alias("Agents_uniques"),
            pl.col(col_sal).mean().round(0).alias("Salaire_moyen"),
        ])
        .sort(["Statut", col_sexe])
    )
    print(stats_sexe)

Hors barème par sexe :
shape: (4, 5)
┌────────────────────────────────┬──────────┬───────────┬────────────────┬───────────────┐
│ Statut                         ┆ SEXE_STD ┆ Nb_lignes ┆ Agents_uniques ┆ Salaire_moyen │
│ ---                            ┆ ---      ┆ ---       ┆ ---            ┆ ---           │
│ str                            ┆ str      ┆ u32       ┆ u32            ┆ f64           │
╞════════════════════════════════╪══════════╪═══════════╪════════════════╪═══════════════╡
│ Inférieur au minimum du barème ┆ Femme    ┆ 667568    ┆ 51433          ┆ 272362.0      │
│ Inférieur au minimum du barème ┆ Homme    ┆ 825161    ┆ 63302          ┆ 284513.0      │
│ Supérieur au maximum du barème ┆ Femme    ┆ 64683     ┆ 48219          ┆ 1.534601e6    │
│ Supérieur au maximum du barème ┆ Homme    ┆ 298585    ┆ 86942          ┆ 840383.0      │
└────────────────────────────────┴──────────┴───────────┴────────────────┴───────────────┘


## 10. Agents extrêmes — Cas les plus incohérents

**Identifier les cas avec les écarts les plus importants pour comprendre les causes.**

In [15]:
# Top 20 des agents les plus en dessous du minimum
if "ecart_pct" in hors.columns:
    cols_detail = [c for c in [col_id, "mois_annee", col_sal, "min_bareme", "max_bareme",
                               "ecart_fcfa", "ecart_pct", "GRADE", "GRADE_SOURCE",
                               "bareme", col_sexe, "emploi", "organisme"]
                   if c and c in hors.columns]
    
    print("=" * 60)
    print("TOP 20 — PLUS INFÉRIEURS AU MINIMUM")
    print("=" * 60)
    print(hors.sort("ecart_pct").select(cols_detail).head(20))
    
    print("\n" + "=" * 60)
    print("TOP 20 — PLUS SUPÉRIEURS AU MAXIMUM")
    print("=" * 60)
    print(hors.sort("ecart_pct", descending=True).select(cols_detail).head(20))

TOP 20 — PLUS INFÉRIEURS AU MINIMUM
shape: (20, 13)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬──────────┬───────────┬───────────┐
│ matricule ┆ mois_anne ┆ montant_b ┆ min_barem ┆ … ┆ bareme    ┆ SEXE_STD ┆ emploi    ┆ organisme │
│ ---       ┆ e         ┆ rut_barem ┆ e         ┆   ┆ ---       ┆ ---      ┆ ---       ┆ ---       │
│ str       ┆ ---       ┆ e         ┆ ---       ┆   ┆ str       ┆ str      ┆ str       ┆ str       │
│           ┆ str       ┆ ---       ┆ f64       ┆   ┆           ┆          ┆           ┆           │
│           ┆           ┆ f64       ┆           ┆   ┆           ┆          ┆           ┆           │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪══════════╪═══════════╪═══════════╡
│ 057426N   ┆ 092024    ┆ 0.0       ┆ 646210.0  ┆ … ┆ Barême    ┆ Homme    ┆ Administr ┆ Min.      │
│           ┆           ┆           ┆           ┆   ┆ Général   ┆          ┆ ateur     ┆ d'Etat    │
│           ┆           ┆           ┆  

## 11. Hypothèses d'incohérence

**Tester les causes possibles du hors barème.**

In [16]:
# HYPOTHÈSE 1 : Grades imputés (pas observés) → erreur de grade ?
if "GRADE_SOURCE" in hors.columns:
    print("=" * 60)
    print("HYPOTHÈSE 1 : Grade observé vs imputé")
    print("=" * 60)
    print("Si un grade est IMPUTÉ, le barème associé peut être incorrect.\n")
    
    h1 = (
        hors.group_by(["Statut", "GRADE_SOURCE"])
        .agg([
            pl.len().alias("Nb_lignes"),
            pl.col(col_id).n_unique().alias("Agents_uniques"),
        ])
        .sort(["Statut", "GRADE_SOURCE"])
    )
    print(h1)
    
    # Comparer avec la base complète
    h1_base = (
        base.group_by("GRADE_SOURCE")
        .agg(pl.len().alias("Nb_lignes_total"))
    )
    h1_hors = (
        hors.group_by("GRADE_SOURCE")
        .agg(pl.len().alias("Nb_hors"))
    )
    comparaison = h1_base.join(h1_hors, on="GRADE_SOURCE", how="left").with_columns(
        (pl.col("Nb_hors").fill_null(0) / pl.col("Nb_lignes_total") * 100).round(1).alias("Taux_hors_%")
    )
    print("\nTaux hors barème par source de grade :")
    print(comparaison)

HYPOTHÈSE 1 : Grade observé vs imputé
Si un grade est IMPUTÉ, le barème associé peut être incorrect.

shape: (4, 4)
┌────────────────────────────────┬──────────────┬───────────┬────────────────┐
│ Statut                         ┆ GRADE_SOURCE ┆ Nb_lignes ┆ Agents_uniques │
│ ---                            ┆ ---          ┆ ---       ┆ ---            │
│ str                            ┆ str          ┆ u32       ┆ u32            │
╞════════════════════════════════╪══════════════╪═══════════╪════════════════╡
│ Inférieur au minimum du barème ┆ IMPUTE       ┆ 232474    ┆ 13858          │
│ Inférieur au minimum du barème ┆ OBSERVE      ┆ 1260255   ┆ 105543         │
│ Supérieur au maximum du barème ┆ IMPUTE       ┆ 2654      ┆ 447            │
│ Supérieur au maximum du barème ┆ OBSERVE      ┆ 360614    ┆ 134737         │
└────────────────────────────────┴──────────────┴───────────┴────────────────┘

Taux hors barème par source de grade :
shape: (2, 4)
┌──────────────┬─────────────────┬──────

In [17]:
# HYPOTHÈSE 2 : Salaires à 0 ou très faibles → erreur de saisie ?
print("=" * 60)
print("HYPOTHÈSE 2 : Salaires nuls ou très faibles")
print("=" * 60)

if col_sal:
    n_zero = int((hors[col_sal] == 0).sum())
    n_negatif = int((hors[col_sal] < 0).sum())
    n_tres_faible = int((hors[col_sal] < 10000).sum())  # < 10 000 FCFA
    
    print(f"Salaire = 0         : {n_zero:,}")
    print(f"Salaire < 0         : {n_negatif:,}")
    print(f"Salaire < 10 000    : {n_tres_faible:,}")
    print(f"\n→ Si beaucoup de salaires à 0, c'est probablement des agents "
          "en congé/suspension pas filtrés.")

HYPOTHÈSE 2 : Salaires nuls ou très faibles
Salaire = 0         : 6,171
Salaire < 0         : 0
Salaire < 10 000    : 6,232

→ Si beaucoup de salaires à 0, c'est probablement des agents en congé/suspension pas filtrés.


In [18]:
# HYPOTHÈSE 3 : Multi-postes → salaire cumulé dépasse le barème ?
if "NB_POSTES" in hors.columns:
    print("=" * 60)
    print("HYPOTHÈSE 3 : Multi-postes et hors barème")
    print("=" * 60)
    
    h3 = (
        hors.group_by(["Statut", "NB_POSTES"])
        .agg([
            pl.len().alias("Nb_lignes"),
            pl.col(col_id).n_unique().alias("Agents_uniques"),
            pl.col(col_sal).mean().round(0).alias("Salaire_moyen"),
        ])
        .sort(["Statut", "NB_POSTES"])
    )
    print(h3)
    print("\n→ Si les multi-postes sont surreprésentés chez les 'Supérieurs', "
          "le cumul de postes explique le dépassement.")

HYPOTHÈSE 3 : Multi-postes et hors barème
shape: (5, 5)
┌────────────────────────────────┬───────────┬───────────┬────────────────┬───────────────┐
│ Statut                         ┆ NB_POSTES ┆ Nb_lignes ┆ Agents_uniques ┆ Salaire_moyen │
│ ---                            ┆ ---       ┆ ---       ┆ ---            ┆ ---           │
│ str                            ┆ u32       ┆ u32       ┆ u32            ┆ f64           │
╞════════════════════════════════╪═══════════╪═══════════╪════════════════╪═══════════════╡
│ Inférieur au minimum du barème ┆ 1         ┆ 1492153   ┆ 114642         ┆ 278876.0      │
│ Inférieur au minimum du barème ┆ 2         ┆ 553       ┆ 88             ┆ 811927.0      │
│ Inférieur au minimum du barème ┆ 3         ┆ 23        ┆ 1              ┆ 654914.0      │
│ Supérieur au maximum du barème ┆ 1         ┆ 362827    ┆ 134884         ┆ 961695.0      │
│ Supérieur au maximum du barème ┆ 2         ┆ 441       ┆ 284            ┆ 2.855552e6    │
└───────────────────────

In [19]:
# HYPOTHÈSE 4 : Problème de matching barème (fuzzy join incorrect)
if "bareme" in hors.columns:
    print("=" * 60)
    print("HYPOTHÈSE 4 : Matching barème potentiellement incorrect")
    print("=" * 60)
    
    n_sans_bareme = int(hors["bareme"].is_null().sum())
    print(f"Agents hors barème SANS barème assigné : {n_sans_bareme:,} "
          f"({100*n_sans_bareme/len(hors):.1f}%)")
    
    if "emploi" in hors.columns:
        # Top emplois hors barème sans barème
        sans_bareme = hors.filter(pl.col("bareme").is_null())
        if len(sans_bareme) > 0:
            top_emplois = (
                sans_bareme
                .group_by("emploi")
                .agg(pl.len().alias("Nb"))
                .sort("Nb", descending=True)
                .head(15)
            )
            print("\nTop 15 emplois hors barème SANS barème assigné :")
            print(top_emplois)

HYPOTHÈSE 4 : Matching barème potentiellement incorrect
Agents hors barème SANS barème assigné : 0 (0.0%)


In [20]:
# HYPOTHÈSE 5 : Le référentiel ANSTAT est trop restrictif ?
# Comparer les bornes du référentiel avec les salaires réels des CONFORMES
print("=" * 60)
print("HYPOTHÈSE 5 : Comparaison bornes référentiel vs salaires réels")
print("=" * 60)

if "GRADE" in base.columns and "min_bareme" in base.columns:
    comparaison_bornes = (
        base.filter(pl.col(col_sal).is_not_null() & (pl.col(col_sal) > 0))
        .group_by("GRADE")
        .agg([
            pl.col(col_sal).min().round(0).alias("Salaire_reel_min"),
            pl.col(col_sal).max().round(0).alias("Salaire_reel_max"),
            pl.col("min_bareme").first().round(0).alias("Borne_ref_min"),
            pl.col("max_bareme").first().round(0).alias("Borne_ref_max"),
            pl.len().alias("Nb_total"),
        ])
        .with_columns([
            (pl.col("Salaire_reel_min") - pl.col("Borne_ref_min")).alias("Ecart_bas"),
            (pl.col("Salaire_reel_max") - pl.col("Borne_ref_max")).alias("Ecart_haut"),
        ])
        .sort("GRADE")
    )
    print(comparaison_bornes)
    print("\n→ Si Ecart_bas très négatif : le référentiel est trop haut par rapport à la réalité.")
    print("→ Si Ecart_haut très positif : le référentiel est trop bas par rapport à la réalité.")

HYPOTHÈSE 5 : Comparaison bornes référentiel vs salaires réels
shape: (12, 8)
┌───────┬─────────────┬─────────────┬─────────────┬────────────┬──────────┬───────────┬────────────┐
│ GRADE ┆ Salaire_ree ┆ Salaire_ree ┆ Borne_ref_m ┆ Borne_ref_ ┆ Nb_total ┆ Ecart_bas ┆ Ecart_haut │
│ ---   ┆ l_min       ┆ l_max       ┆ in          ┆ max        ┆ ---      ┆ ---       ┆ ---        │
│ str   ┆ ---         ┆ ---         ┆ ---         ┆ ---        ┆ u32      ┆ f64       ┆ f64        │
│       ┆ f64         ┆ f64         ┆ f64         ┆ f64        ┆          ┆           ┆            │
╞═══════╪═════════════╪═════════════╪═════════════╪════════════╪══════════╪═══════════╪════════════╡
│ A3    ┆ 2864.0      ┆ 2.7137225e7 ┆ 312834.0    ┆ 570569.0   ┆ 1093849  ┆ -309970.0 ┆ 2.6566656e │
│       ┆             ┆             ┆             ┆            ┆          ┆           ┆ 7          │
│ A4    ┆ 7500.0      ┆ 2.5063237e7 ┆ 340285.0    ┆ 662454.0   ┆ 804391   ┆ -332785.0 ┆ 2.4400783e │
│       ┆    

## 12. Focus sur un agent spécifique

**Remplacez l'identifiant ci-dessous pour explorer un agent en particulier.**

In [21]:
# ⬇️ Remplacez par un matricule/cle_unique trouvé dans les analyses ci-dessus
AGENT_A_EXAMINER = hors[col_id].head(1).item()  # Premier agent hors barème par défaut

print(f"Agent examiné : {AGENT_A_EXAMINER}")
print("=" * 60)

# Historique complet dans la base (tous statuts)
historique = base.filter(pl.col(col_id) == AGENT_A_EXAMINER)

cols_hist = [c for c in [col_id, "mois_annee", "ANNEE", col_sal, "min_bareme", "max_bareme",
                          "Statut", "GRADE", "GRADE_SOURCE", "bareme", "emploi", 
                          "organisme", "NB_POSTES", col_sexe]
             if c and c in historique.columns]

print(f"\nHistorique complet ({len(historique)} mois) :")
print(historique.select(cols_hist).sort("mois_annee"))

Agent examiné : 057426N

Historique complet (3 mois) :
shape: (3, 14)
┌───────────┬────────────┬───────┬────────────┬───┬────────────┬────────────┬───────────┬──────────┐
│ matricule ┆ mois_annee ┆ ANNEE ┆ montant_br ┆ … ┆ emploi     ┆ organisme  ┆ NB_POSTES ┆ SEXE_STD │
│ ---       ┆ ---        ┆ ---   ┆ ut_bareme  ┆   ┆ ---        ┆ ---        ┆ ---       ┆ ---      │
│ str       ┆ str        ┆ i32   ┆ ---        ┆   ┆ str        ┆ str        ┆ u32       ┆ str      │
│           ┆            ┆       ┆ f64        ┆   ┆            ┆            ┆           ┆          │
╞═══════════╪════════════╪═══════╪════════════╪═══╪════════════╪════════════╪═══════════╪══════════╡
│ 057426N   ┆ 012025     ┆ 2025  ┆ 0.0        ┆ … ┆ Administra ┆ Min.       ┆ 1         ┆ Homme    │
│           ┆            ┆       ┆            ┆   ┆ teur Civil ┆ d'Etat Adm ┆           ┆          │
│           ┆            ┆       ┆            ┆   ┆            ┆ inistratio ┆           ┆          │
│           ┆        

## 13. Résumé et pistes d'investigation

Exécutez cette cellule après avoir exploré les résultats ci-dessus.

In [2]:
print("=" * 60)
print("RÉSUMÉ")
print("=" * 60)

n_lignes_hors = len(hors)
n_agents_hors = hors[col_id].n_unique()
n_agents_total_base = base[col_id].n_unique()

print(f"\nLignes hors barème    : {n_lignes_hors:,} (agent × mois)")
print(f"Agents uniques hors   : {n_agents_hors:,}")
print(f"Agents total          : {n_agents_total_base:,}")
print(f"% agents hors barème  : {100*n_agents_hors/n_agents_total_base:.1f}%")

if "ecart_pct" in hors.columns:
    n_faible = int((hors["ecart_pct"].abs() <= 5).sum())
    n_modere = int(((hors["ecart_pct"].abs() > 5) & (hors["ecart_pct"].abs() <= 30)).sum())
    n_fort = int((hors["ecart_pct"].abs() > 30).sum())
    print(f"\nÉcart ≤ 5%  (cas limites)   : {n_faible:,}")
    print(f"Écart 5-30% (modéré)        : {n_modere:,}")
    print(f"Écart > 30% (fort, à vérifier) : {n_fort:,}")

print("\n" + "-" * 60)
print("PISTES D'INVESTIGATION :")
print("-" * 60)
print("1. Grades imputés → le grade assigné est-il correct ?")
print("2. Salaires nuls/très faibles → congés, suspensions non filtrés ?")
print("3. Multi-postes → le cumul dépasse les bornes d'un poste unique ?")
print("4. Matching barème → l'emploi est-il bien associé au bon secteur ?")
print("5. Référentiel trop strict → les bornes ANSTAT sont-elles réalistes ?")
print("6. Erreurs de saisie → montants manifestement aberrants ?")

RÉSUMÉ


NameError: name 'hors' is not defined